In [9]:
import sys
from pathlib import Path

# put src on the path whether the notebook runs from notebooks/ or the repo root
src = Path("../src") if Path("../src").exists() else Path("src")
sys.path.insert(0, str(src.resolve()))

%load_ext autoreload
%autoreload 2

import pandas as pd

from book import BOOK
from data import clean_prices, download_prices, to_returns
from stresstesting import stress

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Data

In [10]:
tickers = [t for desk in BOOK.values() for t in desk]
prices = clean_prices(download_prices(tickers, start_date="2007-01-01"))

# keep partial-NaN rows: FX and equities trade on different calendars
returns = prices.pct_change(fill_method=None).dropna(how="all")

[*********************100%***********************]  9 of 9 completed

Index(['AAPL', 'EURUSD=X', 'GBPUSD=X', 'IEF', 'JPM', 'MSFT', 'SPY', 'TLT',
       'UNH'],
      dtype='str', name='Ticker')
dropped: 0 rows with all missing values


## Stress

In [11]:
results = stress(returns, BOOK)
results

,scenario,desk,worst_1d,worst_10d,total
0,GFC 2008,equity_ls,-1.447101e+06,-2.018618e+06,2.145224e+05
1,GFC 2008,rates_fx,-5.711540e+05,-6.072922e+05,1.969750e+05
2,GFC 2008,BOOK,-1.331656e+06,-1.741492e+06,4.114974e+05
3,COVID 2020,equity_ls,-8.037994e+05,-8.510170e+04,2.346219e+05
4,COVID 2020,rates_fx,-4.733459e+05,-7.437327e+05,3.354493e+05
5,COVID 2020,BOOK,-8.208854e+05,-1.986426e+05,5.700712e+05
6,Rates 2022,equity_ls,-2.932083e+05,-5.761383e+05,9.826686e+05
7,Rates 2022,rates_fx,-1.871229e+05,-5.229305e+05,-1.693118e+06
8,Rates 2022,BOOK,-3.656712e+05,-7.435092e+05,-7.104494e+05


## Results

Replays each crisis's daily returns on today's positions, with no trading. $ P&L: `worst_1d` is the worst day, `worst_10d` the worst 10-day run, `total` the whole window.

- **Equity desk is hedged** against SPY, so crashes hurt less. Its risk is the names moving differently from the index (worst in 2008).
- **Rates/FX desk loses most in 2022** (about -1.7m): long TLT / short IEF is hurt when long rates rise.
- **The desks offset each other**: equity gains in 2022 cut the book's loss to about -0.7m.

Limits: only three past episodes, fixed positions, linear P&L only.
